# Water‑Project Reproducible Notebook
This notebook contains the exact code blocks used to generate plots and checks. Update the paths in **Cell 1** to point to your files.


In [ ]:
# =========================
# 1) CONFIG — EDIT THESE PATHS
# =========================


import os

PROJECT_DIR = "/content/drive/MyDrive/Water-Project"   # <-- edit if needed

# Input files (edit names as needed)
AI_FILE    = os.path.join(PROJECT_DIR, "DC_CONUS.csv")
POWER_FILE = os.path.join(PROJECT_DIR, "Power.xlsx")
TRI_FILE   = os.path.join(PROJECT_DIR, "TRI_2024.csv")

# Common layers (edit if you store elsewhere)
BASINS_SHP = os.path.join(PROJECT_DIR, "hybas_na_lev08_v1c.shp")
RIVERS_SHP = os.path.join(PROJECT_DIR, "HydroRIVERS_v10_na.shp")

print("PROJECT_DIR =", PROJECT_DIR)
print("AI_FILE     =", AI_FILE)
print("POWER_FILE  =", POWER_FILE)
print("TRI_FILE    =", TRI_FILE)
print("BASINS_SHP  =", BASINS_SHP)
print("RIVERS_SHP  =", RIVERS_SHP)


## 2)ECDF (Sector vs Basin-matched Random) + Final Journal Table

**Outputs (saved to** `/`**):**
- ECDF figure PNG
- `TABLE_ECDF_tests_<COL>.csv`
- `TABLE_ECDF_descriptives_<COL>.csv`
- `TABLE_ECDF_FINAL_JOURNAL_<COL>.csv`

**Subtitle (tests included):** two-sample KS (ECDF shape) + MWU (median shift) + Cliff’s δ; Benjamini–Hochberg FDR (q-values).


In [ ]:


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import mannwhitneyu, ks_2samp

PROJECT_DIR = '/content/drive/MyDrive/Water-Project'
OUT_DIR = os.path.join(PROJECT_DIR, 'outputs_ecdf')
os.makedirs(OUT_DIR, exist_ok=True)

COL = 'dist_river_km'  # dist_lake_km / dist_coast_km / dist_any_km
SECTORS = ['AI', 'Power', 'TRI']

SEC_COLORS = {'AI': '#1f77b4', 'Power': '#2ca02c', 'TRI': '#9467bd'}
FIG_OUT = os.path.join(OUT_DIR, f'ECDF_{COL}_JOURNAL.png')

SUBTITLE = (
    'Tests: two-sample Kolmogorov–Smirnov (ECDF shape) + Mann–Whitney U (median shift); '
    'effect size: Cliff’s δ; multiple testing: Benjamini–Hochberg FDR (q).'
)

ECDF_XMAX_Q = 0.995
ECDF_LOG_XMIN = 0.05
SEC_LW = 3.0
RND_LW = 2.0
RND_LS = (0, (1, 2))
RND_A  = 0.95
FIGSIZE = (10.8, 6.4)
DPI_SAVE = 450

def ecdf(a):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a)]
    a = a[a > 0]
    a = np.sort(a)
    if a.size == 0:
        return np.array([np.nan]), np.array([np.nan])
    y = np.arange(1, len(a) + 1) / len(a)
    return a, y

def clean_vec(s):
    v = pd.to_numeric(pd.Series(s), errors='coerce').to_numpy(float)
    v = v[np.isfinite(v)]
    v = v[v > 0]
    return v

def load_dist(sec, grp):
    f = os.path.join(OUT_DIR, f'{sec}_{grp}_distances.csv')
    if not os.path.exists(f):
        raise FileNotFoundError(f'Missing: {f}')
    df = pd.read_csv(f, low_memory=False)
    if COL not in df.columns:
        raise KeyError(f"{f} missing '{COL}'. Has: {list(df.columns)[:30]}")
    return clean_vec(df[COL])

def cliffs_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    n, m = len(x), len(y)
    gt = lt = 0
    chunk = max(1, 2_000_000 // max(1, m))
    for i in range(0, n, chunk):
        xx = x[i:i+chunk]
        diff = xx[:, None] - y[None, :]
        gt += np.sum(diff > 0)
        lt += np.sum(diff < 0)
    return float((gt - lt) / (n*m))

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    idx = np.where(np.isfinite(pvals))[0]
    if len(idx) == 0:
        return out
    pv = pvals[idx]
    order = np.argsort(pv)
    pv_sorted = pv[order]
    n = len(pv_sorted)
    q = pv_sorted * n / (np.arange(1, n+1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out[idx[order]] = np.clip(q, 0, 1)
    return out

def qnt(a, p):
    return float(np.quantile(a, p)) if len(a) else np.nan

def fmt_num(x, nd=2):
    if pd.isna(x):
        return ''
    try:
        x = float(x)
        if abs(x) >= 1000:
            return f'{x:,.0f}'
        return f'{x:.{nd}f}'
    except:
        return str(x)

def fmt_p(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s.startswith('<') or 'e-' in s.lower():
        return s
    try:
        v = float(x)
        if v < 1e-4: return '<1e-4'
        if v < 0.001: return '<0.001'
        return f'{v:.3f}'
    except:
        return s

# ------------------ Load data
data = {}
all_vals = []
for sec in SECTORS:
    x = load_dist(sec, 'sector')
    y = load_dist(sec, 'random')
    data[sec] = {'sector': x, 'random': y}
    all_vals.append(x); all_vals.append(y)
    print(f"{sec}: N_sector={len(x):,} | N_random={len(y):,}")

all_vals = np.concatenate(all_vals) if len(all_vals) else np.array([])
xmax = float(np.quantile(all_vals, ECDF_XMAX_Q)) if all_vals.size else 1.0

# ------------------ ECDF plot
plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 160,
    'savefig.dpi': DPI_SAVE,
    'font.size': 12,
    'axes.linewidth': 1.1,
    'xtick.major.width': 1.1,
    'ytick.major.width': 1.1,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
})

fig, ax = plt.subplots(figsize=FIGSIZE)
for sec in SECTORS:
    xs, ys = ecdf(data[sec]['sector'])
    xr, yr = ecdf(data[sec]['random'])
    c = SEC_COLORS[sec]
    ax.plot(xs, ys, color=c, lw=SEC_LW, ls='-', alpha=1.0)
    ax.plot(xr, yr, color=c, lw=RND_LW, ls=RND_LS, alpha=RND_A)

ax.set_xscale('log')
ax.set_xlim(ECDF_LOG_XMIN, max(1.0, xmax))
ax.set_ylim(0, 1.02)

title_main = 'ECDF Distance to Major Rivers — Sector vs Basin-matched Random'
ax.set_title(title_main, pad=18, fontweight='semibold')
ax.text(0.5, 1.02, SUBTITLE, ha='center', va='bottom', transform=ax.transAxes, fontsize=10.5)

ax.set_xlabel('Distance to major rivers (km, log scale) — HydroRIVERS ORD_STRA ≥ 5')
ax.set_ylabel('ECDF')

n_text = 'N matched (in basins):\n' + '\n'.join([f"{sec}: {len(data[sec]['sector']):,}" for sec in SECTORS])
ax.text(0.02, 0.98, n_text, transform=ax.transAxes, ha='left', va='top', fontsize=10.5,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor='black', linewidth=1.0, alpha=0.95))

style_handles = [
    Line2D([0],[0], color='black', lw=SEC_LW, ls='-', label='Sector'),
    Line2D([0],[0], color='black', lw=RND_LW, ls=RND_LS, label='Random (N-matched)'),
]
color_handles = [
    Line2D([0],[0], color=SEC_COLORS['AI'], lw=3.2, label='AI'),
    Line2D([0],[0], color=SEC_COLORS['Power'], lw=3.2, label='Power'),
    Line2D([0],[0], color=SEC_COLORS['TRI'], lw=3.2, label='TRI'),
]
fig.legend(handles=style_handles + color_handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=11,
           handlelength=3.0, columnspacing=1.8)

ax.grid(True, which='major', linewidth=0.6, alpha=0.18)
ax.grid(False, which='minor')
ax.set_axisbelow(True)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(FIG_OUT, bbox_inches='tight')
plt.show()
print('✅ Saved:', FIG_OUT)

# ------------------ Stats tables
test_rows = []
desc_rows = []

for sec in SECTORS:
    x = data[sec]['sector']
    y = data[sec]['random']

    desc_rows.append({'Distance_Column': COL, 'Sector': sec, 'Group': 'Sector',
                      'N': int(len(x)), 'Median': qnt(x, 0.50),
                      'IQR_25': qnt(x, 0.25), 'IQR_75': qnt(x, 0.75),
                      'Mean': float(np.mean(x)) if len(x) else np.nan})
    desc_rows.append({'Distance_Column': COL, 'Sector': sec, 'Group': 'Random',
                      'N': int(len(y)), 'Median': qnt(y, 0.50),
                      'IQR_25': qnt(y, 0.25), 'IQR_75': qnt(y, 0.75),
                      'Mean': float(np.mean(y)) if len(y) else np.nan})

    if len(x) and len(y):
        ksD, ksP = ks_2samp(x, y, alternative='two-sided', mode='auto')
        u, p_u = mannwhitneyu(x, y, alternative='two-sided')
        cd = cliffs_delta(x, y)
        med_diff = qnt(x, 0.50) - qnt(y, 0.50)
    else:
        ksD = ksP = u = p_u = cd = med_diff = np.nan

    test_rows.append({
        'Distance_Column': COL, 'Sector': sec,
        'N_sector': int(len(x)), 'N_random': int(len(y)),
        'KS_D': float(ksD), 'KS_p': float(ksP),
        'MWU_U': float(u), 'MWU_p': float(p_u),
        'Cliffs_delta': float(cd),
        'Median_diff_sector_minus_random': float(med_diff),
    })

tests = pd.DataFrame(test_rows)
descs = pd.DataFrame(desc_rows)

tests['KS_q_FDR'] = bh_fdr(tests['KS_p'].values)
tests['MWU_q_FDR'] = bh_fdr(tests['MWU_p'].values)

OUT_TESTS = os.path.join(OUT_DIR, f'TABLE_ECDF_tests_{COL}.csv')
OUT_DESCS = os.path.join(OUT_DIR, f'TABLE_ECDF_descriptives_{COL}.csv')
tests.to_csv(OUT_TESTS, index=False)
descs.to_csv(OUT_DESCS, index=False)
print('✅ Saved:', OUT_TESTS)
print('✅ Saved:', OUT_DESCS)

# ------------------ Final wide journal table
sector_desc = descs[descs['Group'] == 'Sector'].rename(columns={
    'N': 'N_sector', 'Median': 'Median_sector', 'IQR_25': 'IQR25_sector',
    'IQR_75': 'IQR75_sector', 'Mean': 'Mean_sector'
}).drop(columns=['Group'])

random_desc = descs[descs['Group'] == 'Random'].rename(columns={
    'N': 'N_random', 'Median': 'Median_random', 'IQR_25': 'IQR25_random',
    'IQR_75': 'IQR75_random', 'Mean': 'Mean_random'
}).drop(columns=['Group'])

final = (sector_desc
         .merge(random_desc, on=['Distance_Column', 'Sector'], how='left', validate='one_to_one')
         .merge(tests, on=['Distance_Column', 'Sector', 'N_sector', 'N_random'], how='left', validate='one_to_one'))

final_disp = final.copy()
final_disp['Median (sector)'] = final_disp['Median_sector'].apply(lambda x: fmt_num(x, 2))
final_disp['IQR (sector)'] = final_disp.apply(lambda r: f"{fmt_num(r['IQR25_sector'],2)}–{fmt_num(r['IQR75_sector'],2)}", axis=1)
final_disp['Median (random)'] = final_disp['Median_random'].apply(lambda x: fmt_num(x, 2))
final_disp['IQR (random)'] = final_disp.apply(lambda r: f"{fmt_num(r['IQR25_random'],2)}–{fmt_num(r['IQR75_random'],2)}", axis=1)

final_disp['KS D'] = final_disp['KS_D'].apply(lambda x: fmt_num(x, 3))
final_disp['KS p'] = final_disp['KS_p'].apply(fmt_p)
final_disp['KS q(FDR)'] = final_disp['KS_q_FDR'].apply(fmt_p)
final_disp['MWU p'] = final_disp['MWU_p'].apply(fmt_p)
final_disp['MWU q(FDR)'] = final_disp['MWU_q_FDR'].apply(fmt_p)
final_disp["Cliff's δ"] = final_disp['Cliffs_delta'].apply(lambda x: fmt_num(x, 3))
final_disp['Δ median (sector-random)'] = final_disp['Median_diff_sector_minus_random'].apply(lambda x: fmt_num(x, 2))

final_table = final_disp[[
    'Sector',
    'N_sector', 'Median (sector)', 'IQR (sector)',
    'N_random', 'Median (random)', 'IQR (random)',
    'KS D', 'KS p', 'KS q(FDR)',
    'MWU p', 'MWU q(FDR)',
    "Cliff's δ", 'Δ median (sector-random)'
]].copy()

OUT_FINAL = os.path.join(OUT_DIR, f'TABLE_ECDF_FINAL_JOURNAL_{COL}.csv')
final_table.to_csv(OUT_FINAL, index=False)
print('✅ Saved:', OUT_FINAL)

display(final_table)


## 3) Aqueduct stress join verification map (2 panels, left margin info + legend)

In [ ]:
!pip -q install geopandas pyogrio rtree shapely contextily

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
from shapely.geometry import Point
import contextily as cx
from shapely.geometry import Point

PROJECT_DIR  = PROJECT_DIR
AI_FILE      = AI_FILE
BASINS_SHP   = BASINS_SHP
AQUEDUCT_CSV = os.path.join(PROJECT_DIR, "Aqueduct40_baseline_monthly_y2023m07d05.csv")

OUT_MAP = os.path.join(PROJECT_DIR, "verify_AqueductStress_join_2panel.png")

PROJ_CRS   = "EPSG:5070"
WEB_CRS    = "EPSG:3857"
STRESS_COL = "bws_01_raw"

AI_INDEX   = 0
REGION_KM  = 150
RADIUS_KM  = 10

PLOT_LOCAL_RANDOM = True
N_LOCAL_RANDOM    = 2500
SEED = 7

def pick_col(cols, keys):
    cols_l = {str(c).lower(): c for c in cols}
    for k in keys:
        if k in cols_l:
            return cols_l[k]
    for c in cols:
        cl = str(c).lower()
        if any(k in cl for k in keys):
            return c
    return None

def load_points_csv(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.astype(str).str.strip()
    lat = pick_col(df.columns, ["latitude","lat","y"])
    lon = pick_col(df.columns, ["longitude","lon","lng","long","x"])
    if lat is None or lon is None:
        raise RuntimeError(f"Could not find lat/lon. cols[:60]={list(df.columns)[:60]}")
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df = df.dropna(subset=[lon, lat]).copy()
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326")

def sample_points_in_circle(x0, y0, r, n, rng):
    ang = rng.uniform(0, 2*np.pi, n)
    rad = r * np.sqrt(rng.uniform(0, 1, n))
    xs = x0 + rad * np.cos(ang)
    ys = y0 + rad * np.sin(ang)
    return gpd.GeoDataFrame(geometry=gpd.points_from_xy(xs, ys), crs=PROJ_CRS)

def add_north_arrow(ax, x=0.95, y=0.88, size=0.08):
    ax.annotate("N", xy=(x, y), xytext=(x, y-size),
                xycoords=ax.transAxes, textcoords=ax.transAxes,
                ha="center", va="center",
                arrowprops=dict(arrowstyle="-|>", lw=1.6))

def add_scalebar(ax, label="5 km", x0=0.08, y0=0.06, x1=0.26, lw=5):
    ax.plot([x0, x1], [y0, y0], transform=ax.transAxes, lw=lw, color="k", solid_capstyle="butt")
    ax.text(x0, y0+0.02, label, transform=ax.transAxes, ha="left", va="bottom",
            fontsize=10, weight="bold")

for p in [AI_FILE, BASINS_SHP, AQUEDUCT_CSV]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}")

ai = load_points_csv(AI_FILE).to_crs(PROJ_CRS)
if AI_INDEX >= len(ai):
    raise ValueError(f"AI_INDEX={AI_INDEX} too large. len(AI)={len(ai)}")

ai_one = ai.iloc[[AI_INDEX]].copy()
x0, y0 = float(ai_one.geometry.iloc[0].x), float(ai_one.geometry.iloc[0].y)

bas = gpd.read_file(BASINS_SHP, engine="pyogrio")
bas.columns = bas.columns.astype(str).str.strip()
if bas.crs is None:
    bas = bas.set_crs("EPSG:4326")
if "PFAF_ID" not in bas.columns:
    raise RuntimeError("Expected PFAF_ID in basins shapefile.")
bas["pfaf6"] = (pd.to_numeric(bas["PFAF_ID"], errors="coerce") // 100).astype("Int64")

aq = pd.read_csv(AQUEDUCT_CSV, low_memory=False)
aq.columns = aq.columns.astype(str).str.strip()
if ("pfaf_id" not in aq.columns) or (STRESS_COL not in aq.columns):
    raise RuntimeError("Aqueduct CSV missing pfaf_id or bws_01_raw.")
aq_small = aq[["pfaf_id", STRESS_COL]].copy()
aq_small["pfaf_id"] = pd.to_numeric(aq_small["pfaf_id"], errors="coerce").astype("Int64")
aq_small[STRESS_COL] = pd.to_numeric(aq_small[STRESS_COL], errors="coerce")

bas2 = bas[["pfaf6","geometry"]].merge(aq_small, left_on="pfaf6", right_on="pfaf_id", how="left")
bas2 = bas2.rename(columns={STRESS_COL:"stress"}).drop(columns=["pfaf_id"], errors="ignore")
bas2 = bas2.dropna(subset=["stress"]).copy().to_crs(PROJ_CRS)

stress_base = bas2["stress"].to_numpy(dtype=float)
t1, t2 = np.quantile(stress_base, [1/3, 2/3]).astype(float)

bas2["stress_bin"] = pd.cut(
    bas2["stress"],
    bins=[-np.inf, t1, t2, np.inf],
    labels=["Low","Medium","High"],
    include_lowest=True
)

ai_one_w = gpd.sjoin(ai_one, bas2[["pfaf6","stress","stress_bin","geometry"]], how="left", predicate="within")
if ai_one_w["pfaf6"].isna().all():
    raise RuntimeError("Selected AI point not in a stress basin. Try another AI_INDEX.")

pfaf6     = int(ai_one_w["pfaf6"].iloc[0])
ai_stress = float(ai_one_w["stress"].iloc[0])
ai_bin    = str(ai_one_w["stress_bin"].iloc[0])

bas_one = bas2.loc[bas2["pfaf6"] == pfaf6].copy()

region_buf = Point(x0, y0).buffer(REGION_KM*1000.0)
bas_region = bas2[bas2.intersects(region_buf)].copy()

circle10 = gpd.GeoDataFrame(geometry=[Point(x0, y0).buffer(RADIUS_KM*1000.0)], crs=PROJ_CRS)

local_pts = None
if PLOT_LOCAL_RANDOM:
    rng = np.random.default_rng(SEED)
    cand = sample_points_in_circle(x0, y0, RADIUS_KM*1000.0, N_LOCAL_RANDOM, rng)
    local_pts = gpd.sjoin(cand, bas2[["stress_bin","geometry"]], how="left", predicate="within").dropna()

bin_order = ["Low","Medium","High"]
bin_to_int = {b:i for i,b in enumerate(bin_order)}
cmap = mpl.colors.ListedColormap(["#2b8cbe", "#41ab5d", "#de2d26"])
norm = mpl.colors.BoundaryNorm([0,1,2,3], cmap.N)
bas_region["bin_i"] = bas_region["stress_bin"].map(bin_to_int).astype(int)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(16.8, 8.4))
fig.subplots_adjust(left=0.33, right=0.98, bottom=0.06, top=0.92, wspace=0.06)

bas_region_web = bas_region.to_crs(WEB_CRS)
bas_one_web    = bas_one.to_crs(WEB_CRS)
ai_web         = ai_one.to_crs(WEB_CRS)

bas_region_web.plot(ax=axA, column="bin_i", cmap=cmap, norm=norm,
                    linewidth=0.35, edgecolor="white", alpha=0.72)
bas_one_web.plot(ax=axA, facecolor="none", edgecolor="black", linewidth=2.8, zorder=4)
ai_web.plot(ax=axA, color="red", markersize=95, edgecolor="black", linewidth=1.2, zorder=5)
cx.add_basemap(axA, source=cx.providers.CartoDB.Positron)
axA.set_axis_off()
axA.set_title("A) Regional Aqueduct water-stress bins (HydroBASINS)\nSelected basin outlined; AI facility shown",
              weight="bold", fontsize=14)
add_north_arrow(axA)
add_scalebar(axA, "20 km")

bas_region.to_crs(WEB_CRS).plot(ax=axB,
                               column=bas_region["stress_bin"].map(bin_to_int).astype(int),
                               cmap=cmap, norm=norm, linewidth=0.25, edgecolor="white", alpha=0.32)
circle10.to_crs(WEB_CRS).plot(ax=axB, facecolor="none", edgecolor="red", linewidth=2.3, zorder=4)
ai_web.plot(ax=axB, color="red", markersize=105, edgecolor="black", linewidth=1.2, zorder=5)

if PLOT_LOCAL_RANDOM and local_pts is not None and len(local_pts) > 0:
    local_web = local_pts.to_crs(WEB_CRS)
    local_web["bin_i"] = local_web["stress_bin"].map(bin_to_int).astype(int)
    local_web.plot(ax=axB, markersize=9, column="bin_i", cmap=cmap, norm=norm, alpha=0.55, linewidth=0)

minx, miny, maxx, maxy = circle10.to_crs(WEB_CRS).total_bounds
pad = 2500
axB.set_xlim(minx-pad, maxx+pad)
axB.set_ylim(miny-pad, maxy+pad)

cx.add_basemap(axB, source=cx.providers.CartoDB.Positron)
axB.set_axis_off()
axB.set_title(f"B) Local verification within {RADIUS_KM:.0f} km\nStress bins + {RADIUS_KM:.0f} km radius",
              weight="bold", fontsize=14)
add_north_arrow(axB)
add_scalebar(axB, "5 km")

info = (
    f"PFAF6={pfaf6}\n"
    f"Aqueduct stress={ai_stress:.3f}\n"
    f"Bin={ai_bin}\n\n"
    f"GLOBAL tertile cutoffs:\n"
    f"Low < {t1:.3f}\n"
    f"Medium < {t2:.3f}\n"
    f"High ≥ {t2:.3f}"
)
fig.text(0.02, 0.86, info, ha="left", va="top",
         bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black", alpha=0.98),
         fontsize=12, weight="bold")

handles = [
    mpl.lines.Line2D([0],[0], marker="s", linestyle="None", markerfacecolor=cmap(0),
                     markeredgecolor="black", markersize=12, label="Low"),
    mpl.lines.Line2D([0],[0], marker="s", linestyle="None", markerfacecolor=cmap(1),
                     markeredgecolor="black", markersize=12, label="Medium"),
    mpl.lines.Line2D([0],[0], marker="s", linestyle="None", markerfacecolor=cmap(2),
                     markeredgecolor="black", markersize=12, label="High"),
    mpl.lines.Line2D([0],[0], marker="o", linestyle="None", markerfacecolor="red",
                     markeredgecolor="black", markersize=10, label="AI facility"),
    mpl.lines.Line2D([0],[0], color="black", lw=2.8, label=f"Selected basin (PFAF6={pfaf6})"),
]
fig.legend(handles=handles,
           loc="upper left",
           bbox_to_anchor=(0.02, 0.34),
           frameon=True, framealpha=0.98,
           fontsize=12,
           title=None)

plt.savefig(OUT_MAP, dpi=350, bbox_inches="tight")
plt.show()

print("✅ Saved:", OUT_MAP)
print(f"AI_INDEX={AI_INDEX} | PFAF6={pfaf6} | stress={ai_stress:.6f} | bin={ai_bin}")
print(f"Tertile cutoffs: t1={t1:.6f}, t2={t2:.6f}")


## 4) Full pipeline — AI + Power + TRI logistic regression vs N‑matched random (major rivers)

In [ ]:
!pip -q install geopandas pyogrio rtree shapely openpyxl scipy statsmodels

import os, warnings, numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from shapely.strtree import STRtree
import statsmodels.api as sm

warnings.filterwarnings("ignore")

AI_FILE    = AI_FILE
POWER_FILE = POWER_FILE
TRI_FILE   = TRI_FILE
BASINS_SHP = BASINS_SHP
RIVERS_SHP = RIVERS_SHP

OUT_DIR    = os.path.join(PROJECT_DIR, "outputs_logit_allsectors_river")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_TABLE  = os.path.join(OUT_DIR, "logit_results_AI_Power_TRI_vs_random_majorRivers.csv")
OUT_FIG    = os.path.join(OUT_DIR, "predicted_probability_curves_AI_Power_TRI_vs_random_majorRivers.png")

ORD_COL = "ORD_STRA"
STREAM_ORDER_MIN = 5
PROJ_CRS = "EPSG:5070"
SIMPLIFY_RIVER_M = 200

SEED = 7
SEED_MAP = {"AI": 101, "Power": 202, "TRI": 303}
RANDOM_BATCH = 40000
NEAREST_CHUNK = 20000

def pick_col(cols, keys):
    cols_l = {str(c).lower(): c for c in cols}
    for k in keys:
        if k in cols_l:
            return cols_l[k]
    for c in cols:
        cl = str(c).lower()
        if any(k in cl for k in keys):
            return c
    return None

def safe_sjoin(left_gdf, right_gdf, how="left", predicate="within", rsuffix="bas"):
    L = left_gdf.copy()
    R = right_gdf.copy()
    for c in ["index_right", "index_left", f"index_{rsuffix}", "index_bas"]:
        if c in L.columns: L = L.drop(columns=[c], errors="ignore")
        if c in R.columns: R = R.drop(columns=[c], errors="ignore")
    L = L.reset_index(drop=True)
    R = R.reset_index(drop=True)
    return gpd.sjoin(L, R, how=how, predicate=predicate, rsuffix=rsuffix)

def generate_random_points_in_basins(bas_gdf_proj, n, seed=0, batch=40000, max_tries=700):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = bas_gdf_proj.total_bounds
    kept = []
    total = 0
    tries = 0
    while total < n and tries < max_tries:
        tries += 1
        xs = rng.uniform(minx, maxx, batch)
        ys = rng.uniform(miny, maxy, batch)
        pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xs, ys), crs=bas_gdf_proj.crs)
        j = safe_sjoin(pts, bas_gdf_proj[["geometry"]], how="inner", predicate="within", rsuffix="bas")
        if len(j) == 0:
            continue
        need = n - total
        kept.append(j.iloc[:need].copy())
        total += len(kept[-1])
    if total < n:
        raise RuntimeError(f"Random generation failed: got {total}/{n}")
    out = pd.concat(kept, ignore_index=True).iloc[:n].copy()
    return gpd.GeoDataFrame(out, geometry="geometry", crs=bas_gdf_proj.crs)

def build_tree(geoms):
    geoms = list(geoms)
    return STRtree(geoms), geoms

def nearest_dist_km(points_proj, tree, target_geoms, chunk=20000):
    pts = list(points_proj)
    out = np.empty(len(pts), dtype=float)
    for start in range(0, len(pts), chunk):
        sub = pts[start:start+chunk]
        nearest = tree.nearest(sub)
        if len(nearest) > 0 and isinstance(nearest[0], (int, np.integer)):
            near_geoms = [target_geoms[i] for i in nearest]
        else:
            near_geoms = nearest
        for i, (p, g) in enumerate(zip(sub, near_geoms)):
            out[start+i] = p.distance(g) / 1000.0
    return out

def load_ai_csv(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.astype(str).str.strip()
    lat = pick_col(df.columns, ["latitude", "lat", "y"])
    lon = pick_col(df.columns, ["longitude", "lon", "lng", "long", "x"])
    if lat is None or lon is None:
        raise RuntimeError(f"AI: could not detect lat/lon in {list(df.columns)[:60]}")
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df = df.dropna(subset=[lon, lat]).copy()
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326")

def load_power_xlsx(path):
    df = pd.read_excel(path, header=1)
    df.columns = df.columns.astype(str).str.strip()
    if "Latitude" not in df.columns or "Longitude" not in df.columns:
        lat = pick_col(df.columns, ["latitude", "lat", "y"])
        lon = pick_col(df.columns, ["longitude", "lon", "lng", "long", "x"])
    else:
        lat, lon = "Latitude", "Longitude"
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df = df.dropna(subset=[lon, lat]).copy()
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326")

def load_tri_csv(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.astype(str).str.strip()
    if "12. LATITUDE" in df.columns and "13. LONGITUDE" in df.columns:
        lat, lon = "12. LATITUDE", "13. LONGITUDE"
    else:
        lat = pick_col(df.columns, ["latitude", "lat", "y"])
        lon = pick_col(df.columns, ["longitude", "lon", "lng", "long", "x"])
    if lat is None or lon is None:
        raise RuntimeError(f"TRI: could not detect lat/lon in {list(df.columns)[:60]}")
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df = df.dropna(subset=[lon, lat]).copy()
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326")

def fit_logit_and_curve(dist_sector_km, dist_random_km):
    df = pd.DataFrame({
        "presence": np.r_[np.ones(len(dist_sector_km), dtype=int),
                          np.zeros(len(dist_random_km), dtype=int)],
        "dist_km":  np.r_[dist_sector_km, dist_random_km]
    })
    df["log_dist"] = np.log(df["dist_km"] + 1.0)
    X = sm.add_constant(df[["log_dist"]])
    y = df["presence"]
    res = sm.Logit(y, X).fit(disp=False)
    coef = float(res.params["log_dist"])
    OR = float(np.exp(coef))
    ci = res.conf_int()
    OR_lo = float(np.exp(ci.loc["log_dist", 0]))
    OR_hi = float(np.exp(ci.loc["log_dist", 1]))
    pval = float(res.pvalues["log_dist"])
    return res, OR, OR_lo, OR_hi, pval, df

need = [AI_FILE, POWER_FILE, TRI_FILE, BASINS_SHP, RIVERS_SHP]
missing = [p for p in need if not os.path.exists(p)]
if missing:
    raise FileNotFoundError("Missing inputs:\n" + "\n".join(missing))

bas = gpd.read_file(BASINS_SHP, engine="pyogrio")
bas.columns = bas.columns.astype(str).str.strip()
if bas.crs is None:
    bas = bas.set_crs("EPSG:4326")
bas = bas.to_crs("EPSG:4326")
bas_proj = bas.to_crs(PROJ_CRS)

rivers = gpd.read_file(RIVERS_SHP, engine="pyogrio")
rivers.columns = rivers.columns.astype(str).str.strip()
if ORD_COL not in rivers.columns:
    raise RuntimeError(f"Expected {ORD_COL} in HydroRIVERS.")
rivers[ORD_COL] = pd.to_numeric(rivers[ORD_COL], errors="coerce")
rmaj = rivers.loc[rivers[ORD_COL].notna() & (rivers[ORD_COL] >= STREAM_ORDER_MIN), ["geometry"]].copy()
print(f"✅ Major river segments (ORD≥{STREAM_ORDER_MIN}): {len(rmaj):,}")

rmaj_proj = rmaj.to_crs(PROJ_CRS).copy()
rmaj_proj["geometry"] = rmaj_proj["geometry"].simplify(SIMPLIFY_RIVER_M)
tree_riv, geoms_riv = build_tree(rmaj_proj.geometry)

ai  = load_ai_csv(AI_FILE);          ai["sector"]="AI"
pwr = load_power_xlsx(POWER_FILE);   pwr["sector"]="Power"
tri = load_tri_csv(TRI_FILE);        tri["sector"]="TRI"
sectors = [("AI", ai), ("Power", pwr), ("TRI", tri)]

results_rows = []
curves = {}

for sec, gdf in sectors:
    gdf_ll = gdf.to_crs(bas.crs)
    g_in = safe_sjoin(gdf_ll, bas[["geometry"]], how="inner", predicate="within", rsuffix="bas")
    N = len(g_in)
    print(f"✅ {sec} points inside basins: {N:,}")
    if N == 0:
        continue

    seed = SEED + SEED_MAP.get(sec, 999)
    rd = generate_random_points_in_basins(bas_proj, n=N, seed=seed, batch=RANDOM_BATCH)

    g_proj = g_in.to_crs(PROJ_CRS)
    dist_sec  = nearest_dist_km(g_proj.geometry, tree_riv, geoms_riv, chunk=NEAREST_CHUNK)
    dist_rand = nearest_dist_km(rd.geometry,     tree_riv, geoms_riv, chunk=NEAREST_CHUNK)

    res, OR, OR_lo, OR_hi, pval, df_model = fit_logit_and_curve(dist_sec, dist_rand)

    dmax = float(np.nanpercentile(df_model["dist_km"], 99.5))
    dist_grid = np.linspace(0, max(50.0, dmax), 300)
    log_grid = np.log(dist_grid + 1.0)
    X_pred = sm.add_constant(pd.DataFrame({"log_dist": log_grid}))
    prob = res.predict(X_pred)
    curves[sec] = {"dist_grid": dist_grid, "prob": prob}

    results_rows.append({
        "sector": sec,
        "N_sector": int(N),
        "N_random": int(N),
        "coef_log_dist": float(res.params["log_dist"]),
        "OR_log_dist": OR,
        "OR_2.5%": OR_lo,
        "OR_97.5%": OR_hi,
        "p_value": pval,
        "pseudo_R2": float(res.prsquared),
        "LLR_pvalue": float(res.llr_pvalue),
    })

df_out = pd.DataFrame(results_rows).sort_values("sector")
df_out.to_csv(OUT_TABLE, index=False)
print("\n✅ Saved logit results:", OUT_TABLE)

plt.figure(figsize=(8.2, 5.8))
for sec in ["AI", "Power", "TRI"]:
    if sec in curves:
        plt.plot(curves[sec]["dist_grid"], curves[sec]["prob"], linewidth=2, label=sec)

plt.xlabel(f"Distance to Major Rivers (km) — HydroRIVERS {ORD_COL} ≥ {STREAM_ORDER_MIN}")
plt.ylabel("Predicted Probability of Sector Presence\n(vs N-matched random)")
plt.title("Predicted Probability Curves from Logistic Regression")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_FIG, dpi=350)
plt.show()

print("✅ Saved probability curve figure:", OUT_FIG)


## 5) Stallings Center — nearest water (single location) + proof map

In [ ]:


import os, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import fiona

from shapely.ops import nearest_points

warnings.filterwarnings("ignore")

try:
    import contextily as cx
except:
    !pip -q install contextily
    import contextily as cx

PROJECT_DIR = PROJECT_DIR
OUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

DC_PATH    = os.path.join(PROJECT_DIR, "Stalling.csv")

RIVERS_SHP = "/content/drive/MyDrive/Copy of HydroRIVERS_v10_na.shp"
COAST_SHP  = os.path.join(PROJECT_DIR, "ne_10m_coastline.shp")
LAKES_GDB  = os.path.join(PROJECT_DIR, "_HYDROLAKES_GDB", "HydroLAKES_polys_v10.gdb")

OUT_CSV = os.path.join(OUT_DIR, "stalling_nearest_water.csv")
OUT_MAP = os.path.join(OUT_DIR, "stalling_proof_map.png")

RIVER_STREAMORDER_THRESHOLD = 3
STREAM_ORDER_COL_CANDIDATES = ["ORD_STRA","ORD_FLOW","STREAMORDE","STRM_ORDER","ORD_ST"]

def detect_lat_lon_columns(df):
    cols = {c.lower().strip(): c for c in df.columns}
    return cols["latitude"], cols["longitude"]

def utm_epsg_from_lonlat(lon, lat):
    zone = int(np.floor((lon + 180) / 6) + 1)
    return 32600 + zone

def nearest_bruteforce(point_geom, gdf):
    if len(gdf) == 0: return np.nan, None
    dmin, gmin = np.inf, None
    for g in gdf.geometry.values:
        d = point_geom.distance(g)
        if d < dmin: dmin, gmin = d, g
    _, near = nearest_points(point_geom, gmin)
    return float(dmin), near

print("Loading Stalling.csv…")
dc = pd.read_csv(DC_PATH)
lat_col, lon_col = detect_lat_lon_columns(dc)

lon = float(dc.iloc[0][lon_col])
lat = float(dc.iloc[0][lat_col])
print("Point:", lat, lon)

print("Loading water layers…")
rivers = gpd.read_file(RIVERS_SHP).to_crs("EPSG:4326")
coast  = gpd.read_file(COAST_SHP).to_crs("EPSG:4326")
layers = fiona.listlayers(LAKES_GDB)
lakes = gpd.read_file(LAKES_GDB, layer=layers[0]).to_crs("EPSG:4326")

stream_col = next(c for c in STREAM_ORDER_COL_CANDIDATES if c in rivers.columns)
rivers = rivers[rivers[stream_col] >= RIVER_STREAMORDER_THRESHOLD]

utm = utm_epsg_from_lonlat(lon, lat)
p = gpd.GeoSeries(gpd.points_from_xy([lon],[lat]), crs="EPSG:4326").to_crs(f"EPSG:{utm}").iloc[0]
rivers_utm = rivers.to_crs(f"EPSG:{utm}")
lakes_utm  = lakes.to_crs(f"EPSG:{utm}")
coast_utm  = coast.to_crs(f"EPSG:{utm}")

d_r, _ = nearest_bruteforce(p, rivers_utm)
d_l, _ = nearest_bruteforce(p, lakes_utm.boundary)
d_c, _ = nearest_bruteforce(p, coast_utm)

nearest = min([("River",d_r),("Lake",d_l),("Coast",d_c)], key=lambda x: x[1])

out = {
    "latitude": lat,
    "longitude": lon,
    "d_river_km": d_r/1000,
    "d_lake_km": d_l/1000,
    "d_coast_km": d_c/1000,
    "nearest": nearest[0]
}

pd.DataFrame([out]).to_csv(OUT_CSV, index=False)
print("Saved CSV:", OUT_CSV)

p_web = gpd.GeoSeries([p], crs=f"EPSG:{utm}").to_crs(3857)

fig, ax = plt.subplots(figsize=(8,8))
cx.add_basemap(ax)
p_web.plot(ax=ax, color="red", markersize=120)

ax.set_title("Stallings Center — Location")
ax.axis("off")

plt.savefig(OUT_MAP, dpi=300)
plt.close()

print("Saved map:", OUT_MAP)
print("DONE ✅")


## 6) HUC2 3‑panel choropleth — share of national facilities (%)

In [ ]:

!pip -q install geopandas pyogrio rtree shapely fiona openpyxl

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import fiona
from matplotlib.patches import Patch
import matplotlib.patheffects as pe

PROJECT_DIR  = PROJECT_DIR
AI_FILE      = AI_FILE
POWER_FILE   = POWER_FILE
TRI_FILE     = TRI_FILE
HUC_FILE     = os.path.join(PROJECT_DIR, "WBD_National_GPKG", "WBD_National_GPKG.gpkg")

OUT_FIG      = os.path.join(PROJECT_DIR, "HUC2_FacilitySharePct_3panel_AI_Power_TRI_BBOXLABELS.png")

N_LABELS = 12
LABEL_MIN_PCT = 0.20
CANDIDATE_POOL = 80

LABEL_FONTSIZE = 7.4
LABEL_WEIGHT = "semibold"
TEXT_EFFECTS = [pe.withStroke(linewidth=2.0, foreground="white")]

EDGE_COLOR = "black"
EDGE_LW = 1.0

NUDGE_RINGS = [0, 60000, 110000, 160000, 220000]
DIRS = [
    (0, 0),
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (1, 1), (-1, 1), (1, -1), (-1, -1),
    (2, 0), (-2, 0), (0, 2), (0, -2)
]

BINS = [0, 0.000001, 1, 3, 6, 10, 100]
BIN_LABELS = ["0%", "0–1%", "1–3%", "3–6%", "6–10%", ">10%"]

def pick_col(cols, keys):
    cols_l = {str(c).lower(): c for c in cols}
    for k in keys:
        if k in cols_l:
            return cols_l[k]
    for c in cols:
        cl = str(c).lower()
        if any(k in cl for k in keys):
            return c
    return None

def find_excel_header_row(path, sheet_name=None, max_rows=80):
    xls = pd.ExcelFile(path)
    sheets = [sheet_name] if sheet_name else xls.sheet_names
    for sh in sheets:
        preview = pd.read_excel(path, sheet_name=sh, header=None, nrows=max_rows)
        for r in range(len(preview)):
            row_vals = preview.iloc[r].astype(str).str.lower().tolist()
            has_lat = any(("latitude" in v) or (v.strip() == "lat") for v in row_vals)
            has_lon = any(("longitude" in v) or (v.strip() in ["lon","long","lng"]) for v in row_vals)
            if has_lat and has_lon:
                return r, sh
    raise RuntimeError("Could not find Latitude/Longitude header row in Power.xlsx")

def load_points_csv(path, name, force_lat=None, force_lon=None):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.astype(str).str.strip()

    if force_lat and force_lon and force_lat in df.columns and force_lon in df.columns:
        lat, lon = force_lat, force_lon
    else:
        lat = pick_col(df.columns, ["lat","latitude","y"])
        lon = pick_col(df.columns, ["lon","longitude","lng","long","x"])

    if lat is None or lon is None:
        raise RuntimeError(f"{name}: could not detect lat/lon. Columns: {df.columns[:80].tolist()}")

    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df = df.dropna(subset=[lat, lon]).copy()

    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326")

def load_points_power_excel(path):
    header_row, sheet_used = find_excel_header_row(path)
    print(f"✅ Power.xlsx header row={header_row}, sheet='{sheet_used}'")

    df = pd.read_excel(path, sheet_name=sheet_used, header=header_row)
    df.columns = df.columns.astype(str).str.strip()

    lat = pick_col(df.columns, ["latitude","lat"])
    lon = pick_col(df.columns, ["longitude","lon","long","lng"])
    if lat is None or lon is None:
        raise RuntimeError(f"Power: could not detect lat/lon. Columns: {df.columns[:80].tolist()}")

    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df = df.dropna(subset=[lat, lon]).copy()

    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326")

def pct_classify(pct_series):
    v = pd.Series(pct_series).astype(float)
    cats = pd.cut(v, bins=BINS, labels=BIN_LABELS, include_lowest=True, right=True).astype(str)
    cats[v == 0] = "0%"
    return cats

def make_huc2_pct_map(points_gdf, huc2_gdf, HUC2_ID):
    j = gpd.sjoin(points_gdf, huc2_gdf[[HUC2_ID, "geometry"]], how="inner", predicate="within")
    counts = j.groupby(HUC2_ID).size().rename("count").reset_index()

    m = huc2_gdf[[HUC2_ID, "geometry"]].merge(counts, on=HUC2_ID, how="left")
    m["count"] = m["count"].fillna(0).astype(int)

    total = int(m["count"].sum())
    m["pct"] = 0.0 if total == 0 else (100.0 * m["count"] / total)
    m["class"] = pct_classify(m["pct"])
    return m, total

def legend_handles(color_map):
    return [Patch(facecolor=color_map[lbl], edgecolor=EDGE_COLOR, label=lbl) for lbl in BIN_LABELS]

def _bboxes_intersect(b1, b2, pad=2.0):
    return not (b1[2] + pad < b2[0] or b1[0] - pad > b2[2] or b1[3] + pad < b2[1] or b1[1] - pad > b2[3])

def place_labels_bbox(ax, gdf, n_labels, min_pct, candidate_pool):
    fig = ax.figure
    renderer = fig.canvas.get_renderer()

    candidates = (
        gdf[gdf["pct"] >= min_pct]
        .sort_values("pct", ascending=False)
        .head(candidate_pool)
        .copy()
    )

    kept = 0
    kept_bboxes = []

    for _, row in candidates.iterrows():
        if kept >= n_labels:
            break

        rp = row.geometry.representative_point()
        base_x, base_y = float(rp.x), float(rp.y)
        label = f"{float(row['pct']):.1f}%"
        placed = False

        for r in NUDGE_RINGS:
            for dx, dy in DIRS:
                x = base_x + dx * r
                y = base_y + dy * r

                t = ax.text(
                    x, y, label,
                    ha="center", va="center",
                    fontsize=LABEL_FONTSIZE,
                    fontweight=LABEL_WEIGHT,
                    color="black",
                    path_effects=TEXT_EFFECTS,
                    zorder=10
                )

                fig.canvas.draw()
                bb = t.get_window_extent(renderer=renderer)
                bbox = (bb.x0, bb.y0, bb.x1, bb.y1)

                collision = any(_bboxes_intersect(bbox, bprev) for bprev in kept_bboxes)
                if collision:
                    t.remove()
                else:
                    kept_bboxes.append(bbox)
                    kept += 1
                    placed = True
                    break

            if placed:
                break

        if not placed:
            continue

def draw_panel(ax, gdf, color_map, panel_letter, title, total):
    for lbl in BIN_LABELS:
        sub = gdf[gdf["class"] == lbl]
        if len(sub) > 0:
            sub.plot(ax=ax, color=color_map[lbl], edgecolor=EDGE_COLOR, linewidth=EDGE_LW)

    ax.text(0.01, 0.99, panel_letter, transform=ax.transAxes,
            ha="left", va="top", fontsize=13, fontweight="bold")
    ax.set_title(title, fontsize=13, pad=6)

    place_labels_bbox(ax, gdf, N_LABELS, LABEL_MIN_PCT, CANDIDATE_POOL)

    ax.text(0.99, 0.01, f"N={total:,}", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=9)
    ax.set_axis_off()

layers = fiona.listlayers(HUC_FILE)
if "WBDHU2" not in layers:
    raise RuntimeError(f"WBDHU2 not found. Layers: {layers}")

huc2 = gpd.read_file(HUC_FILE, layer="WBDHU2")
HUC2_ID = pick_col(huc2.columns, ["huc2"])
if HUC2_ID is None:
    raise RuntimeError(f"Could not find HUC2 id field. Columns: {huc2.columns.tolist()}")

huc2[HUC2_ID] = huc2[HUC2_ID].astype(str).str.zfill(2)
huc2 = huc2[~huc2[HUC2_ID].isin(["19","20","21","22"])].copy()
huc2 = huc2.to_crs("EPSG:5070")

ai  = load_points_csv(AI_FILE, "AI").to_crs("EPSG:5070")
pwr = load_points_power_excel(POWER_FILE).to_crs("EPSG:5070")
tri = load_points_csv(TRI_FILE, "TRI", force_lat="12. LATITUDE", force_lon="13. LONGITUDE").to_crs("EPSG:5070")

print("✅ Loaded:", "AI", len(ai), "| Power", len(pwr), "| TRI", len(tri))

m_ai,  tot_ai  = make_huc2_pct_map(ai,  huc2, HUC2_ID)
m_pwr, tot_pwr = make_huc2_pct_map(pwr, huc2, HUC2_ID)
m_tri, tot_tri = make_huc2_pct_map(tri, huc2, HUC2_ID)

blue_map = {
    "0%":"#E6E6E6", "0–1%":"#DEEBF7", "1–3%":"#9ECAE1", "3–6%":"#6BAED6", "6–10%":"#3182BD", ">10%":"#08519C"
}
orng_map = {
    "0%":"#E6E6E6", "0–1%":"#FEE6CE", "1–3%":"#FDBE85", "3–6%":"#FD8D3C", "6–10%":"#E6550D", ">10%":"#A63603"
}
grn_map = {
    "0%":"#E6E6E6", "0–1%":"#E5F5E0", "1–3%":"#A1D99B", "3–6%":"#74C476", "6–10%":"#31A354", ">10%":"#006D2C"
}

plt.rcParams.update({"font.family": "DejaVu Sans"})
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.2), dpi=450)

draw_panel(axes[0], m_ai,  blue_map, "a", "AI Data Centers",  tot_ai)
draw_panel(axes[1], m_pwr, orng_map, "b", "Power Facilities", tot_pwr)
draw_panel(axes[2], m_tri, grn_map,  "c", "TRI Facilities",   tot_tri)

axes[0].legend(handles=legend_handles(blue_map), title="Share of national facilities (%)",
               loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=9, title_fontsize=9)
axes[1].legend(handles=legend_handles(orng_map), title="Share of national facilities (%)",
               loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=9, title_fontsize=9)
axes[2].legend(handles=legend_handles(grn_map),  title="Share of national facilities (%)",
               loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=9, title_fontsize=9)

plt.subplots_adjust(bottom=0.24, wspace=0.08)
plt.savefig(OUT_FIG, dpi=600, bbox_inches="tight")
plt.show()

print("✅ Saved:", OUT_FIG)
print(f"Labels: up to {N_LABELS} per panel | min {LABEL_MIN_PCT:.2f}% | candidates {CANDIDATE_POOL}")


## 7) NWM v3.0 hydro‑regime pipeline (one cell) + ECDF + optional odds ratios

In [ ]:
!pip -q install "geopandas>=0.14" pyogrio shapely rtree \
               "fsspec==2026.2.0" "s3fs==2026.2.0" \
               xarray zarr "dask[complete]" \
               pandas openpyxl scikit-learn matplotlib pyarrow

import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import xarray as xr
import s3fs
from dask.diagnostics import ProgressBar

PROJECT_DIR = PROJECT_DIR
AI_FILE    = AI_FILE
POWER_FILE = POWER_FILE
TRI_FILE   = TRI_FILE
BASINS_SHP = BASINS_SHP

OUTDIR = os.path.join(PROJECT_DIR, "NWM_HydroRegime_Output")
os.makedirs(OUTDIR, exist_ok=True)

START_DATE = "2003-01-01"
END_DATE   = "2020-12-31"
ZARR_PATH = "s3://noaa-nwm-retrospective-3-0-pds/zarr/CONUS/chrtout.zarr"

REACHES_PER_BASIN = 1
TIME_CHUNK = 720
FEAT_CHUNK = 20000
MAP_CHUNK  = 200000

def find_lat_lon_columns(df):
    cols = list(df.columns)
    cols_low = [str(c).strip().lower() for c in cols]
    lat_col = None
    lon_col = None
    for c, cl in zip(cols, cols_low):
        if cl in {"lat", "latitude", "y"}:
            lat_col = c
        if cl in {"lon", "long", "longitude", "x"}:
            lon_col = c
    if lat_col is None:
        for c, cl in zip(cols, cols_low):
            if "latitude" in cl or cl.startswith("lat"):
                lat_col = c
                break
    if lon_col is None:
        for c, cl in zip(cols, cols_low):
            if "longitude" in cl or " long" in cl or cl.startswith("lon"):
                lon_col = c
                break
    return lat_col, lon_col

def load_points_csv_robust(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.astype(str).str.strip()
    lat_col, lon_col = find_lat_lon_columns(df)
    if lat_col is None or lon_col is None:
        raise ValueError(f"Could not detect lat/lon in {path}. Columns sample: {list(df.columns)[:80]}")
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df = df.dropna(subset=[lat_col, lon_col]).copy()
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs=4326)
    print(f"Loaded {os.path.basename(path)} lat='{lat_col}' lon='{lon_col}' rows={len(gdf):,}")
    return gdf

def load_power_excel_fixed(path):
    xls = pd.ExcelFile(path, engine="openpyxl")
    sheet = None
    for s in xls.sheet_names:
        if "plant" in s.lower():
            sheet = s
            break
    if sheet is None:
        sheet = xls.sheet_names[0]
    df = pd.read_excel(path, sheet_name=sheet, header=1, engine="openpyxl")
    df.columns = df.columns.astype(str).str.strip()
    lat_col, lon_col = find_lat_lon_columns(df)
    if lat_col is None or lon_col is None:
        raise ValueError(f"Could not detect lat/lon in Power sheet '{sheet}'. Columns sample: {list(df.columns)[:80]}")
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df = df.dropna(subset=[lat_col, lon_col]).copy()
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs=4326)
    print(f"Loaded Power.xlsx sheet='{sheet}' lat='{lat_col}' lon='{lon_col}' rows={len(gdf):,}")
    return gdf

def assign_basins(points_gdf, basins_gdf, basin_id_field):
    j = gpd.sjoin(points_gdf, basins_gdf[[basin_id_field, "geometry"]], how="left", predicate="within")
    j = j.drop(columns=[c for c in j.columns if c.startswith("index_")], errors="ignore")
    return j

def ecdf_vals(x):
    x = np.asarray(pd.to_numeric(pd.Series(x), errors="coerce").dropna())
    if x.size == 0:
        return np.array([]), np.array([])
    x = np.sort(x)
    y = np.arange(1, x.size + 1) / x.size
    return x, y

def median_iqr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan, np.nan, np.nan
    return s.quantile(0.50), s.quantile(0.25), s.quantile(0.75)

def fmt_median_iqr(med, q1, q3, nd=3):
    if pd.isna(med):
        return ""
    return f"{med:.{nd}f} [{q1:.{nd}f}, {q3:.{nd}f}]"

need = [AI_FILE, POWER_FILE, TRI_FILE, BASINS_SHP]
missing = [p for p in need if not os.path.exists(p)]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

bas = gpd.read_file(BASINS_SHP, engine="pyogrio").to_crs(4326)
id_field = "HYBAS_ID" if "HYBAS_ID" in bas.columns else None
if id_field is None:
    raise ValueError(f"Could not find HYBAS_ID in basins. Columns sample: {list(bas.columns)[:40]}")
area_field = "SUB_AREA" if "SUB_AREA" in bas.columns else None

ai_gdf    = load_points_csv_robust(AI_FILE)
tri_gdf   = load_points_csv_robust(TRI_FILE)
power_gdf = load_power_excel_fixed(POWER_FILE)

ai_gdf    = assign_basins(ai_gdf, bas, id_field).dropna(subset=[id_field]).copy()
tri_gdf   = assign_basins(tri_gdf, bas, id_field).dropna(subset=[id_field]).copy()
power_gdf = assign_basins(power_gdf, bas, id_field).dropna(subset=[id_field]).copy()

ai_gdf[id_field]    = ai_gdf[id_field].astype(np.int64)
tri_gdf[id_field]   = tri_gdf[id_field].astype(np.int64)
power_gdf[id_field] = power_gdf[id_field].astype(np.int64)

def basin_presence_counts(gdf, basin_id_field, prefix):
    out = gdf.groupby(basin_id_field).size().rename(f"n_{prefix}").to_frame().reset_index()
    out[f"{prefix}_present"] = 1
    return out

ai_b = basin_presence_counts(ai_gdf, id_field, "AI")
pw_b = basin_presence_counts(power_gdf, id_field, "Power")
tr_b = basin_presence_counts(tri_gdf, id_field, "TRI")

bas_master = bas[[id_field]].drop_duplicates().copy()
if area_field:
    bas_master["area_km2"] = pd.to_numeric(bas[area_field], errors="coerce")

bm = bas_master.merge(ai_b, on=id_field, how="left") \
               .merge(pw_b, on=id_field, how="left") \
               .merge(tr_b, on=id_field, how="left")

for c in ["AI_present","Power_present","TRI_present"]:
    bm[c] = bm[c].fillna(0).astype(int)
for c in ["n_AI","n_Power","n_TRI"]:
    bm[c] = bm[c].fillna(0).astype(int)

PRES_OUT = os.path.join(OUTDIR, "basin_presence_counts.csv")
bm.to_csv(PRES_OUT, index=False)
print("Saved:", PRES_OUT)

print("\nOpening NWM Zarr (metadata only)...")
fs = s3fs.S3FileSystem(anon=True)
mapper = s3fs.S3Map(root=ZARR_PATH, s3=fs, check=False)
try:
    ds = xr.open_zarr(mapper, consolidated=True)
except Exception:
    ds = xr.open_zarr(mapper, consolidated=False)

ds = ds.chunk({"time": TIME_CHUNK, "feature_id": FEAT_CHUNK})

MAP_FILE = os.path.join(OUTDIR, "feature_to_basin.parquet")
if os.path.exists(MAP_FILE):
    print("\nLoading cached reach->basin map:", MAP_FILE)
    feat2bas = pd.read_parquet(MAP_FILE)
else:
    print("\nBuilding reach->basin map (ONE-TIME heavy step)...")
    nfeat = ds["feature_id"].shape[0]
    maps = []
    for i0 in range(0, nfeat, MAP_CHUNK):
        i1 = min(i0 + MAP_CHUNK, nfeat)
        print(f"  mapping reaches {i0:,}..{i1:,}")
        fid  = ds["feature_id"].isel(feature_id=slice(i0, i1)).values.astype(np.int64)
        lat  = ds["latitude"].isel(feature_id=slice(i0, i1)).values.astype(float)
        lon  = ds["longitude"].isel(feature_id=slice(i0, i1)).values.astype(float)
        ordv = ds["order"].isel(feature_id=slice(i0, i1)).values.astype(float)

        pts = gpd.GeoDataFrame(
            {"feature_id": fid, "order": ordv},
            geometry=gpd.points_from_xy(lon, lat),
            crs=4326
        )

        j = gpd.sjoin(pts, bas[[id_field, "geometry"]], how="left", predicate="within")
        j = j.drop(columns=[c for c in j.columns if c.startswith("index_")], errors="ignore")
        j = j.dropna(subset=[id_field])
        j[id_field] = j[id_field].astype(np.int64)
        maps.append(j[["feature_id", id_field, "order"]])

    feat2bas = pd.concat(maps, ignore_index=True).drop_duplicates(subset=["feature_id"])
    feat2bas.to_parquet(MAP_FILE, index=False)
    print("Saved:", MAP_FILE)

feat2bas_sorted = feat2bas.sort_values("order", ascending=False)
top_reach = feat2bas_sorted.groupby(id_field, as_index=False).head(REACHES_PER_BASIN).copy()
sel_features = top_reach["feature_id"].astype(np.int64).unique()

print("\nSubsetting streamflow to time window + selected reaches...")
ds_sub = ds.sel(time=slice(START_DATE, END_DATE))
fid_index = pd.Index(ds_sub["feature_id"].values.astype(np.int64))
keep_pos = fid_index.get_indexer(sel_features)
keep_pos = keep_pos[keep_pos >= 0]
ds_sub = ds_sub.isel(feature_id=keep_pos)

q = ds_sub["streamflow"]
q_daily = q.resample(time="1D").mean()

q_mean = q_daily.mean("time")
q_std  = q_daily.std("time")
q_cv   = q_std / xr.where(q_mean == 0, np.nan, q_mean)
q_q10 = q_daily.quantile(0.10, dim="time")
q_q90 = q_daily.quantile(0.90, dim="time")

dq = q_daily.diff("time").abs()
rbi = dq.sum("time") / xr.where(q_daily.sum("time") == 0, np.nan, q_daily.sum("time"))

q_mclim = q_daily.groupby("time.month").mean("time")
top3 = q_mclim.sortby(q_mclim, ascending=False).isel(month=slice(0,3)).sum("month")
ann  = q_mclim.sum("month")
season_conc = top3 / xr.where(ann == 0, np.nan, ann)

feat_metrics = xr.Dataset({
    "meanQ": q_mean,
    "stdQ": q_std,
    "CVQ": q_cv,
    "Q10": q_q10,
    "Q90": q_q90,
    "RBI": rbi,
    "season_conc": season_conc
})

print("\nComputing metrics (heavy)...")
with ProgressBar():
    feat_metrics_df = feat_metrics.compute().to_dataframe().reset_index()

feat_metrics_df["feature_id"] = feat_metrics_df["feature_id"].astype(np.int64)
top_reach_small = top_reach[[id_field, "feature_id"]].copy()
top_reach_small["feature_id"] = top_reach_small["feature_id"].astype(np.int64)

feat_metrics_df = feat_metrics_df.merge(top_reach_small, on="feature_id", how="left").dropna(subset=[id_field])
feat_metrics_df[id_field] = feat_metrics_df[id_field].astype(np.int64)

bas_metrics = feat_metrics_df.groupby(id_field).agg({
    "meanQ": "mean",
    "stdQ": "mean",
    "CVQ": "mean",
    "Q10": "mean",
    "Q90": "mean",
    "RBI": "mean",
    "season_conc": "mean"
}).reset_index()

HYDRO_OUT = os.path.join(OUTDIR, "basin_hydrologic_metrics.csv")
bas_metrics.to_csv(HYDRO_OUT, index=False)
print("Saved:", HYDRO_OUT)

master = bm.merge(bas_metrics, on=id_field, how="left")
MASTER_OUT = os.path.join(OUTDIR, "basin_master_presence_hydro.csv")
master.to_csv(MASTER_OUT, index=False)
print("Saved:", MASTER_OUT)

metrics = ["meanQ", "CVQ", "Q10", "Q90", "RBI", "season_conc"]
metrics = [m for m in metrics if m in master.columns]

groups = {
    "AI basins":     master.loc[master["AI_present"]==1],
    "Non-AI basins": master.loc[master["AI_present"]==0],
    "Power basins":  master.loc[master["Power_present"]==1],
    "TRI basins":    master.loc[master["TRI_present"]==1],
}

rows = []
for m in metrics:
    row = {"Metric": m}
    for gname, gdf in groups.items():
        med, q1, q3 = median_iqr(gdf[m])
        row[gname] = fmt_median_iqr(med, q1, q3, nd=3)
    rows.append(row)

table1 = pd.DataFrame(rows)
TABLE_OUT = os.path.join(OUTDIR, "Table_1_hydro_regime_by_sector.csv")
table1.to_csv(TABLE_OUT, index=False)
print("Saved:", TABLE_OUT)

g_ai   = master.loc[master["AI_present"] == 1].copy()
g_pw   = master.loc[master["Power_present"] == 1].copy()
g_tri  = master.loc[master["TRI_present"] == 1].copy()
g_none = master.loc[
    (master["AI_present"] == 0) &
    (master["Power_present"] == 0) &
    (master["TRI_present"] == 0)
].copy()

plot_metrics = [m for m in ["CVQ", "Q10", "Q90", "RBI", "season_conc"] if m in master.columns]
if len(plot_metrics) == 0:
    plot_metrics = [m for m in ["meanQ", "stdQ"] if m in master.columns]

plt.style.use("default")
plt.rcParams.update({
    "figure.dpi": 220,
    "savefig.dpi": 350,
    "font.size": 12,
    "font.weight": "bold",
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
})

COLORS = {"AI":"#1f77b4","Power":"#2ca02c","TRI":"#9467bd","None":"#000000"}

fig_h = 3.35 * len(plot_metrics)
fig, axes = plt.subplots(len(plot_metrics), 1, figsize=(9.6, fig_h), dpi=220)
if len(plot_metrics) == 1:
    axes = [axes]

for i, m in enumerate(plot_metrics):
    ax = axes[i]
    xa, ya = ecdf_vals(g_ai[m])
    xp, yp = ecdf_vals(g_pw[m])
    xt, yt = ecdf_vals(g_tri[m])
    xn, yn = ecdf_vals(g_none[m])

    if xa.size: ax.plot(xa, ya, lw=3.0, color=COLORS["AI"],    label=f"AI (N={len(g_ai):,})")
    if xp.size: ax.plot(xp, yp, lw=3.0, color=COLORS["Power"], label=f"Power (N={len(g_pw):,})")
    if xt.size: ax.plot(xt, yt, lw=3.0, color=COLORS["TRI"],   label=f"TRI (N={len(g_tri):,})")
    if xn.size: ax.plot(xn, yn, lw=2.4, color=COLORS["None"],  ls="--", alpha=0.85, label=f"None (N={len(g_none):,})")

    ax.set_xlabel(m)
    ax.set_ylabel("ECDF")
    ax.grid(True, which="major", linewidth=0.6, alpha=0.20)
    ax.grid(False, which="minor")
    ax.spines["top"].set_alpha(0.5)
    ax.spines["right"].set_alpha(0.5)

    if i == 0:
        ax.set_title("Hydrologic regime ECDF by basin class (NWM v3.0 retrospective)")
        ax.legend(ncol=2, frameon=False)

plt.tight_layout()
FIG_ECDF = os.path.join(OUTDIR, "Figure_ECDF_hydro_regime_AI_Power_TRI_None.png")
plt.savefig(FIG_ECDF, dpi=350, bbox_inches="tight")
plt.show()
print("Saved:", FIG_ECDF)

model_df = master[["AI_present"] + metrics].dropna().copy()
model_df["AI_present"] = model_df["AI_present"].astype(int)

if len(model_df) > 50 and len(metrics) > 0:
    X = model_df[metrics].values
    y = model_df["AI_present"].values
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=4000))])
    pipe.fit(X, y)

    coef = pipe.named_steps["clf"].coef_[0]
    odds = np.exp(coef)

    or_df = pd.DataFrame({"predictor": metrics, "coef": coef, "odds_ratio": odds}) \
             .sort_values("odds_ratio", ascending=False)

    OR_CSV = os.path.join(OUTDIR, "OddsRatios_AI_present.csv")
    or_df.to_csv(OR_CSV, index=False)
    print("Saved:", OR_CSV)

    plt.figure(figsize=(8.6, 5.0))
    plt.bar(or_df["predictor"], or_df["odds_ratio"])
    plt.axhline(1.0, linestyle="--")
    plt.ylabel("Odds ratio (AI_present)")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, axis="y", alpha=0.25)
    plt.tight_layout()
    FIG_OR = os.path.join(OUTDIR, "Figure_OddsRatios_AI_present.png")
    plt.savefig(FIG_OR, dpi=350, bbox_inches="tight")
    plt.show()
    print("Saved:", FIG_OR)
else:
    print("Skipping odds ratio model (not enough rows after dropna or no metrics).")

print("\nDONE ✅ Outputs in:", OUTDIR)


## 8. ECDF + Matched Random Comparisons
### NWM HydroRegime Output (Reproducible)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, mannwhitneyu
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

OUTDIR = os.path.join(os.getcwd(), "NWM_HydroRegime_Output")
os.makedirs(OUTDIR, exist_ok=True)

INFILE = os.path.join(os.getcwd(), "basin_master_presence_hydro_manual.csv")

master = pd.read_csv(INFILE)
valid = master.dropna(subset=["meanQ"]).copy()

def label_exclusive(row):
    if row.get("AI_present", 0) == 1: return "AI"
    if row.get("Power_present", 0) == 1: return "Power"
    if row.get("TRI_present", 0) == 1: return "TRI"
    return "None"

valid["group_excl"] = valid.apply(label_exclusive, axis=1)

SECTORS = ["AI", "Power", "TRI"]
BASELINE = "None"
METRICS = ["RBI", "season_conc", "CVQ"]

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 18,
    "axes.labelsize": 14,
    "legend.fontsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 1.4
})

COLOR = {"AI": "#1f77b4", "Power": "#ff7f0e", "TRI": "#2ca02c", "None": "0.70"}

def to_num(x):
    return pd.to_numeric(pd.Series(x), errors="coerce").dropna().values

def ecdf_step(values):
    v = np.sort(to_num(values))
    if v.size == 0:
        return None, None
    y = np.arange(1, v.size + 1) / v.size
    return v, y

def median_iqr(values):
    v = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if len(v) == 0:
        return np.nan, np.nan, np.nan
    return float(v.quantile(0.5)), float(v.quantile(0.25)), float(v.quantile(0.75))

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return out
    p = pvals[ok]
    m = p.size
    order = np.argsort(p)
    ranked = p[order]
    adj = ranked * m / (np.arange(1, m + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    out_ok = np.empty_like(p)
    out_ok[order] = adj
    out[ok] = out_ok
    return out

def xlim_quantiles(df, metric, qlo=0.02, qhi=0.98):
    pooled = to_num(df[metric])
    if pooled.size == 0:
        return None
    lo, hi = np.quantile(pooled, [qlo, qhi])
    if not (np.isfinite(lo) and np.isfinite(hi) and hi > lo):
        return None
    return (lo, hi)

def matched_random_bootstrap(df, metric, sector, baseline="None", nboot=800, seed=7, grid_n=250):
    rng = np.random.default_rng(seed)
    g = to_num(df.loc[df["group_excl"] == sector, metric])
    bpool = to_num(df.loc[df["group_excl"] == baseline, metric])
    n = g.size
    if n < 30 or bpool.size < n:
        return None
    allv = np.concatenate([g, bpool])
    x_grid = np.quantile(allv, np.linspace(0.01, 0.99, grid_n))
    g_sorted = np.sort(g)
    g_ecdf = np.searchsorted(g_sorted, x_grid, side="right") / g_sorted.size
    b_ecdfs = np.zeros((nboot, x_grid.size), dtype=float)
    diffs = np.zeros(nboot, dtype=float)
    for i in range(nboot):
        b = rng.choice(bpool, size=n, replace=False)
        b_sorted = np.sort(b)
        b_ecdfs[i] = np.searchsorted(b_sorted, x_grid, side="right") / b_sorted.size
        diffs[i] = np.median(g) - np.median(b)
    b_med = np.median(b_ecdfs, axis=0)
    b_lo  = np.quantile(b_ecdfs, 0.025, axis=0)
    b_hi  = np.quantile(b_ecdfs, 0.975, axis=0)
    d_med = float(np.median(diffs))
    d_lo  = float(np.quantile(diffs, 0.025))
    d_hi  = float(np.quantile(diffs, 0.975))
    p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    p_two = float(min(1.0, p_two))
    return {
        "n": int(n),
        "x_grid": x_grid,
        "g_ecdf": g_ecdf,
        "b_med": b_med,
        "b_lo": b_lo,
        "b_hi": b_hi,
        "d_med": d_med,
        "d_lo": d_lo,
        "d_hi": d_hi,
        "p_two": p_two
    }

fig1, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig1.subplots_adjust(top=0.78, wspace=0.22)

legend_handles = []
legend_labels = []

for ax, metric in zip(axes, METRICS):
    lim = xlim_quantiles(valid, metric, 0.02, 0.98)
    if lim:
        ax.set_xlim(*lim)

    x0, y0 = ecdf_step(valid.loc[valid["group_excl"] == "None", metric])
    if x0 is not None:
        h0 = ax.plot(x0, y0, drawstyle="steps-post", color=COLOR["None"], lw=2.2)[0]
        if metric == METRICS[0]:
            legend_handles.append(h0); legend_labels.append("None")

    for s in SECTORS:
        xs, ys = ecdf_step(valid.loc[valid["group_excl"] == s, metric])
        if xs is None:
            continue
        hs = ax.plot(xs, ys, drawstyle="steps-post", color=COLOR[s], lw=3.0)[0]
        if metric == METRICS[0]:
            legend_handles.append(hs); legend_labels.append(s)

    ax.set_title(metric, pad=10)
    ax.grid(True, alpha=0.18)
    ax.set_xlabel("Value")

axes[0].set_ylabel("Cumulative frequency")

fig1.legend(legend_handles, legend_labels, loc="upper center",
            bbox_to_anchor=(0.5, 0.98), ncol=4, frameon=False)

FIG1_OUT = os.path.join(OUTDIR, "FIG1_ECDF_Sector_vs_Sector.png")
fig1.savefig(FIG1_OUT, dpi=600, bbox_inches="tight")
plt.show()

fig2, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=False, sharey=True)
fig2.subplots_adjust(top=0.90, wspace=0.14, hspace=0.18)

style_handles = [
    Line2D([0],[0], color="k", lw=2.8, linestyle="-", label="Sector"),
    Line2D([0],[0], color="k", lw=2.2, linestyle="--", label="Matched random"),
    Patch(facecolor="0.7", edgecolor="none", alpha=0.18, label="95% CI")
]
fig2.legend(handles=style_handles, loc="upper center",
            bbox_to_anchor=(0.5, 0.98), ncol=3, frameon=False)

effect_rows = []

for i, sector in enumerate(SECTORS):
    for j, metric in enumerate(METRICS):
        ax = axes[i, j]
        out = matched_random_bootstrap(valid, metric, sector, baseline=BASELINE)
        if out is None:
            ax.set_visible(False)
            continue
        xg = out["x_grid"]
        ax.plot(xg, out["g_ecdf"], color=COLOR[sector], lw=3.0)
        ax.plot(xg, out["b_med"], color=COLOR[sector], lw=2.4, linestyle="--", alpha=0.95)
        ax.fill_between(xg, out["b_lo"], out["b_hi"], color=COLOR[sector], alpha=0.14, lw=0)

        lim = xlim_quantiles(valid, metric, 0.02, 0.98)
        if lim:
            ax.set_xlim(*lim)

        if i == 0:
            ax.set_title(metric, pad=8)

        if j == 0:
            ax.set_ylabel(sector)
        else:
            ax.set_ylabel("")

        ax.set_xlabel("")
        ax.grid(True, alpha=0.15)

        effect_rows.append({
            "Sector": sector,
            "Metric": metric,
            "N_sector": out["n"],
            "MedianDiff_sector_minus_matched": out["d_med"],
            "CI95_low": out["d_lo"],
            "CI95_high": out["d_hi"],
            "p_two_bootstrap": out["p_two"]
        })

FIG2_OUT = os.path.join(OUTDIR, "FIG2_Sector_vs_MatchedRandom_ECDF_3x3.png")
fig2.savefig(FIG2_OUT, dpi=600, bbox_inches="tight")
plt.show()

eff = pd.DataFrame(effect_rows)
EFF_OUT = os.path.join(OUTDIR, "TABLE_EffectSizes_SectorMinusMatched.csv")
eff.to_csv(EFF_OUT, index=False)

fig3, ax = plt.subplots(1, 1, figsize=(11, 5))
fig3.subplots_adjust(top=0.82)

x = np.arange(len(METRICS))
offset = {"AI": -0.18, "Power": 0.00, "TRI": 0.18}

for sector in SECTORS:
    sub = eff[eff["Sector"] == sector].set_index("Metric").loc[METRICS].reset_index()
    y = sub["MedianDiff_sector_minus_matched"].values
    lo = sub["CI95_low"].values
    hi = sub["CI95_high"].values
    yerr = np.vstack([y - lo, hi - y])

    ax.errorbar(
        x + offset[sector], y, yerr=yerr,
        fmt="o", capsize=4, markersize=7,
        color=COLOR[sector], label=sector
    )

ax.axhline(0, color="0.25", linestyle="--", lw=1.8)
ax.set_xticks(x)
ax.set_xticklabels(METRICS)
ax.set_ylabel("Median Δ (sector − matched)")
ax.grid(True, axis="y", alpha=0.18)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))

FIG3_OUT = os.path.join(OUTDIR, "FIG3_EffectSizes_AllSectors_Combined.png")
fig3.savefig(FIG3_OUT, dpi=600, bbox_inches="tight")
plt.show()

desc_rows = []
for metric in METRICS:
    for g in ["None"] + SECTORS:
        vals = valid.loc[valid["group_excl"] == g, metric]
        v = to_num(vals)
        med, q1, q3 = median_iqr(vals)
        desc_rows.append({
            "Metric": metric,
            "Group": g,
            "N": int(v.size),
            "Median": med,
            "Q1": q1,
            "Q3": q3
        })

desc = pd.DataFrame(desc_rows)
DESC_OUT = os.path.join(OUTDIR, "TABLE_Descriptives_Median_IQR_N.csv")
desc.to_csv(DESC_OUT, index=False)

test_rows = []
for metric in METRICS:
    base = to_num(valid.loc[valid["group_excl"] == "None", metric])
    for sector in SECTORS:
        a = to_num(valid.loc[valid["group_excl"] == sector, metric])
        if a.size < 30 or base.size < 30:
            continue

        ks = ks_2samp(a, base)
        mw = mannwhitneyu(a, base, alternative="two-sided")

        rng = np.random.default_rng(7)
        nboot = 2000
        diffs = np.empty(nboot, dtype=float)
        for i in range(nboot):
            aa = rng.choice(a, size=a.size, replace=True)
            bb = rng.choice(base, size=base.size, replace=True)
            diffs[i] = np.median(aa) - np.median(bb)

        d = float(np.median(a) - np.median(base))
        lo = float(np.quantile(diffs, 0.025))
        hi = float(np.quantile(diffs, 0.975))

        test_rows.append({
            "Metric": metric,
            "Sector": sector,
            "CompareTo": "None",
            "N_sector": int(a.size),
            "N_none": int(base.size),
            "MedianDiff_sector_minus_none": d,
            "CI95_low": lo,
            "CI95_high": hi,
            "KS_D": float(ks.statistic),
            "KS_p": float(ks.pvalue),
            "MWU_p": float(mw.pvalue)
        })

tests = pd.DataFrame(test_rows)
tests["KS_pFDR"] = bh_fdr(tests["KS_p"].values)
tests["MWU_pFDR"] = bh_fdr(tests["MWU_p"].values)

TESTS_OUT = os.path.join(OUTDIR, "TABLE_Tests_Sector_vs_None_KS_MWU_FDR.csv")
tests.to_csv(TESTS_OUT, index=False)

print("Figure 1:", FIG1_OUT)
print("Figure 2:", FIG2_OUT)
print("Figure 3:", FIG3_OUT)
print("Table A:", DESC_OUT)
print("Table B:", EFF_OUT)
print("Table C:", TESTS_OUT)


## 9.  Table 



In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

NB_NAME = os.path.basename(os.getcwd())

OUTDIR = "/content/drive/MyDrive/Water-Project/NWM_HydroRegime_Output"
os.makedirs(OUTDIR, exist_ok=True)

TABLE_SPECS = [
    ("Table A. Descriptives (Median / IQR / N)", "TABLE_Descriptives_Median_IQR_N.csv", "Metric"),
    ("Table B. Sector − Matched Random (Effect sizes)", "TABLE_EffectSizes_SectorMinusMatched.csv", "Sector"),
    ("Table C. Sector vs None (KS / MWU / FDR)", "TABLE_Tests_Sector_vs_None_KS_MWU_FDR.csv", "Metric"),
]

def nice_num(x, nd=2):
    if pd.isna(x):
        return ""
    try:
        x = float(x)
        if abs(x) >= 1000:
            return f"{x:,.0f}"
        return f"{x:.{nd}f}"
    except Exception:
        return str(x)

def insert_group_headers_and_blank_groupcol(df, group_col):
    df = df.copy()
    df[group_col] = df[group_col].astype(str)

    rows = []
    group_header_rows = []
    current = None

    for _, r in df.iterrows():
        g = r[group_col]
        if g != current:
            hdr = {c: "" for c in df.columns}
            hdr[df.columns[0]] = g
            rows.append(hdr)
            group_header_rows.append(len(rows) - 1)
            current = g

        rr = r.to_dict()
        rr[group_col] = ""
        rows.append(rr)

    out = pd.DataFrame(rows, columns=df.columns)
    return out, group_header_rows

def maybe_shorten_columns(df):
    rename = {
        "MedianDiff_sector_minus_matched": "MedianΔ",
        "MedianDiff_sector_minus_none": "MedianΔ",
        "p_two_bootstrap": "p (2-sided)",
        "CI95_low": "CI95 low",
        "CI95_high": "CI95 high",
        "BaselinePool": "Baseline",
        "N_sector": "N",
        "CompareTo": "Compare",
        "KS_D": "KS D",
        "KS_p": "KS p",
        "KS_pFDR": "KS p(FDR)",
        "MWU_p": "MWU p",
        "MWU_pFDR": "MWU p(FDR)",
    }
    cols = {c: rename[c] for c in df.columns if c in rename}
    return df.rename(columns=cols)

def render_table_png(
    df,
    title,
    out_png,
    group_col=None,
    ndigits=2,
    font_size=14,
    title_size=20,
    row_height=0.060,
    header_height=0.070,
    pad_inches=0.40,
    header_gray="0.90",
    group_gray="0.97",
):
    df = df.copy()
    df = maybe_shorten_columns(df)

    for c in df.columns:
        df[c] = df[c].apply(lambda v: nice_num(v, nd=ndigits))

    group_rows = []
    if group_col and group_col in df.columns:
        df, group_rows = insert_group_headers_and_blank_groupcol(df, group_col)

    nrows, ncols = df.shape
    fig_w = 1.6 + 1.35 * ncols
    fig_h = 1.8 + row_height * (nrows + 2) * 10

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    ax.set_title(title, fontsize=title_size, fontweight="bold", pad=14)

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        colLoc="center",
        loc="upper center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)

    try:
        table.auto_set_column_width(col=list(range(ncols)))
    except Exception:
        pass

    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("0.25")
        cell.set_linewidth(0.8)
        if r == 0:
            cell.set_facecolor(header_gray)
            cell.set_text_props(weight="bold")
            cell.set_linewidth(1.5)
            cell.set_edgecolor("0.0")
            cell.set_height(header_height)
        else:
            cell.set_height(row_height)

    for gr in group_rows:
        rr = gr + 1
        for c in range(ncols):
            cell = table[(rr, c)]
            cell.set_facecolor(group_gray)
            cell.set_linewidth(0.0)
            if c == 0:
                cell.set_text_props(weight="bold", ha="left")
            else:
                cell.get_text().set_text("")

    last_r = nrows
    for c in range(ncols):
        table[(last_r, c)].set_linewidth(2.2)
        table[(last_r, c)].set_edgecolor("0.0")

    plt.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=pad_inches)
    plt.close(fig)
    print("Saved:", out_png)

print("Notebook:", NB_NAME)
print("Working directory:", os.getcwd())
print("OUTDIR:", OUTDIR)

for title, fname, group_col in TABLE_SPECS:
    path = os.path.join(OUTDIR, fname)
    if not os.path.exists(path):
        print("Missing:", path)
        continue

    df = pd.read_csv(path)

    if "Sector" in df.columns:
        cols = ["Sector"] + [c for c in df.columns if c != "Sector"]
        df = df[cols]
    if "Metric" in df.columns:
        cols = ["Metric"] + [c for c in df.columns if c != "Metric"]
        df = df[cols]

    out_png = os.path.join(OUTDIR, fname.replace(".csv", f"_SLIDE_{NB_NAME.replace(' ', '_')}.png"))
    render_table_png(
        df=df,
        title=title,
        out_png=out_png,
        group_col=group_col,
        ndigits=2,
        font_size=14,
        title_size=22,
        row_height=0.060,
        header_height=0.075,
        pad_inches=0.45
    )

print("DONE")

#  10) Water-stress siting (AI / Power / TRI)





In [ ]:
# ============================================================
#  STRESS SITING (AI / Power / TRI)
# BOTH BASELINES IN ONE CELL (GLOBAL + BASIN_MATCHED)

# HOW TO RUN:
# 1) Set PROJECT_DIR below to the folder that contains your input files.
# 2) Ensure the files listed in INPUT FILENAMES exist in that folder.
# 3) Run this cell. Outputs will be created in:
#     
# ============================================================



!pip -q install geopandas pyogrio rtree shapely scipy openpyxl

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import chi2_contingency

# -------------------------
# PATHS (EDIT IF NEEDED)
# -------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/Water-Project")

AI_FILE       = PROJECT_DIR / "DC_CONUS.csv"
POWER_FILE    = PROJECT_DIR / "Power.xlsx"
TRI_FILE      = PROJECT_DIR / "TRI_2024.csv"
BASINS_SHP    = PROJECT_DIR / "hybas_na_lev08_v1c.shp"
AQUEDUCT_CSV  = PROJECT_DIR / "Aqueduct40_baseline_monthly_y2023m07d05.csv"

OUT_DIR = PROJECT_DIR / "analysis4_outputs_repro_BOTH_CLEAN"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# PARAMS
# -------------------------
SECTORS    = ["AI","Power","TRI"]
STRESS_COL = "bws_01_raw"

BASE_SEED = 7
SEED_MAP  = {"AI": 101, "Power": 202, "TRI": 303}
MODE_SEED = {"GLOBAL": 10000, "BASIN_MATCHED": 20000}

POOL_FACTOR = 5.0
POOL_CAP    = 300000
BATCH       = 70000
MAX_TRIES   = 60

SHOW_PERCENT_LABELS = True
PCT_DECIMALS        = 1

# -------------------------
# STYLE (clean + professional)
# -------------------------
mpl.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 350,
    "font.size": 12,
    "axes.titleweight": "semibold",
    "axes.labelweight": "semibold",
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.9,
    "ytick.major.width": 0.9,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "legend.frameon": False,
})

COL3 = {"low":"#0072B2", "medium":"#009E73", "high":"#CC79A7"}
COL4 = {"Q1":"#56B4E9", "Q2":"#009E73", "Q3":"#E69F00", "Q4":"#D55E00"}

need = [AI_FILE, POWER_FILE, TRI_FILE, BASINS_SHP, AQUEDUCT_CSV]
missing = [p for p in need if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(str(p) for p in missing))

def pick_col(cols, keys):
    cols_l = {str(c).lower(): c for c in cols}
    for k in keys:
        if k in cols_l:
            return cols_l[k]
    for c in cols:
        cl = str(c).lower()
        if any(k in cl for k in keys):
            return c
    return None

def safe_sjoin(left_gdf, right_gdf, how="left", predicate="within"):
    L = left_gdf.copy()
    R = right_gdf.copy()
    for c in ["index_right", "index_left"]:
        if c in L.columns: L = L.drop(columns=[c], errors="ignore")
        if c in R.columns: R = R.drop(columns=[c], errors="ignore")
    L = L.reset_index(drop=True)
    R = R.reset_index(drop=True)
    return gpd.sjoin(L, R, how=how, predicate=predicate)

def load_points_any(path, sector_name, force_lat=None, force_lon=None, excel_header=0):
    path = Path(path)
    if str(path).lower().endswith((".xlsx",".xls")):
        df = pd.read_excel(path, header=excel_header)
    else:
        df = pd.read_csv(path, low_memory=False)

    df.columns = df.columns.astype(str).str.strip()

    if force_lat and force_lon:
        lat_col, lon_col = force_lat, force_lon
    else:
        lat_col = pick_col(df.columns, ["latitude","lat","y"])
        lon_col = pick_col(df.columns, ["longitude","lon","lng","long","x"])

    if lat_col is None or lon_col is None:
        raise RuntimeError(f"{sector_name}: could not detect lat/lon columns.")

    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
    df = df.dropna(subset=[lon_col, lat_col]).copy()

    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs="EPSG:4326")
    gdf["sector"] = sector_name
    return gdf

def generate_random_points_in_polygonset(poly_gdf, n, seed, batch=BATCH, max_tries=MAX_TRIES):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = poly_gdf.total_bounds
    kept = []
    total = 0
    tries = 0
    while total < n and tries < max_tries:
        tries += 1
        xs = rng.uniform(minx, maxx, batch)
        ys = rng.uniform(miny, maxy, batch)
        pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xs, ys), crs=poly_gdf.crs)
        j = safe_sjoin(pts, poly_gdf[["geometry"]], how="inner", predicate="within")
        if len(j) == 0:
            continue
        need_now = n - total
        kept.append(j.iloc[:need_now].copy())
        total += len(kept[-1])
    if total < n:
        raise RuntimeError(f"Random generation failed: got {total}/{n} points within polygons.")
    out = pd.concat(kept, ignore_index=True).iloc[:n].copy()
    return gpd.GeoDataFrame(out, geometry="geometry", crs=poly_gdf.crs)

def odds_ratio_2x2(a, b, c, d):
    a,b,c,d = float(a),float(b),float(c),float(d)
    if min(a,b,c,d) == 0:
        a += 0.5; b += 0.5; c += 0.5; d += 0.5
    OR = (a/b) / (c/d)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    z = 1.96
    lo = np.exp(np.log(OR) - z*se)
    hi = np.exp(np.log(OR) + z*se)
    return float(OR), float(lo), float(hi)

def make_global_bins_from_basins(stress_series):
    s = pd.to_numeric(stress_series, errors="coerce")
    t1, t2 = s.quantile([1/3, 2/3]).values
    q1, q2, q3 = s.quantile([0.25, 0.50, 0.75]).values

    def tert(v):
        if pd.isna(v): return np.nan
        if v <= t1: return "low"
        if v <= t2: return "medium"
        return "high"

    def quart(v):
        if pd.isna(v): return np.nan
        if v <= q1: return "Q1"
        if v <= q2: return "Q2"
        if v <= q3: return "Q3"
        return "Q4"

    return s.apply(tert), s.apply(quart)

def stable_sort_for_repro(gdf):
    out = gdf.copy()
    out["_wkb"] = out.geometry.apply(lambda g: g.wkb_hex if g is not None else "")
    out = out.sort_values(["_wkb"], kind="mergesort").drop(columns=["_wkb"])
    return out

def clean_axis(ax):
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.0, 0.5, 1.0])
    ax.grid(True, axis="y", linewidth=0.7, alpha=0.18)
    ax.grid(False, axis="x")
    ax.set_axisbelow(True)
    for side in ["left","bottom","right","top"]:
        ax.spines[side].set_linewidth(0.9)

def plot_grouped_percent_only(ax, classes, obs_counts, rnd_counts, color_map, title):
    obs_total = int(obs_counts.sum())
    rnd_total = int(rnd_counts.sum())
    p_obs = (obs_counts / max(1, obs_total)).values
    p_rnd = (rnd_counts / max(1, rnd_total)).values

    x = np.arange(len(classes))
    w = 0.36

    for i, cls in enumerate(classes):
        col = color_map[cls]
        ax.bar(x[i]-w/2, p_obs[i], width=w, color=col, alpha=0.95, edgecolor="black", linewidth=0.9)
        ax.bar(x[i]+w/2, p_rnd[i], width=w, color=col, alpha=0.55, edgecolor="black", linewidth=0.9)

        if SHOW_PERCENT_LABELS:
            ax.text(x[i]-w/2, p_obs[i]+0.015, f"{p_obs[i]*100:.{PCT_DECIMALS}f}%", ha="center", va="bottom", fontsize=10)
            ax.text(x[i]+w/2, p_rnd[i]+0.015, f"{p_rnd[i]*100:.{PCT_DECIMALS}f}%", ha="center", va="bottom", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(classes)
    ax.set_title(title)
    clean_axis(ax)

# -------------------------
# Load basins + Aqueduct stress
# -------------------------
bas = gpd.read_file(BASINS_SHP, engine="pyogrio")
bas.columns = bas.columns.astype(str).str.strip()
if bas.crs is None:
    bas = bas.set_crs("EPSG:4326")
if "PFAF_ID" not in bas.columns:
    raise RuntimeError("Expected PFAF_ID in HydroBASINS shapefile.")

bas["PFAF_ID"] = pd.to_numeric(bas["PFAF_ID"], errors="coerce").astype("Int64")
bas["pfaf6"] = (bas["PFAF_ID"] // 100).astype("Int64")
bas8 = bas[["PFAF_ID","pfaf6","geometry"]].copy()

aq = pd.read_csv(AQUEDUCT_CSV, low_memory=False)
aq.columns = aq.columns.astype(str).str.strip()
if ("pfaf_id" not in aq.columns) or (STRESS_COL not in aq.columns):
    raise RuntimeError("Aqueduct CSV must contain columns: pfaf_id and bws_01_raw.")

aq_small = aq[["pfaf_id", STRESS_COL]].copy()
aq_small["pfaf_id"] = pd.to_numeric(aq_small["pfaf_id"], errors="coerce").astype("Int64")
aq_small[STRESS_COL] = pd.to_numeric(aq_small[STRESS_COL], errors="coerce")

bas_stress = bas8.merge(aq_small, left_on="pfaf6", right_on="pfaf_id", how="left")
bas_stress = bas_stress.rename(columns={STRESS_COL:"stress_value"}).drop(columns=["pfaf_id"], errors="ignore")

terts, quarts = make_global_bins_from_basins(bas_stress["stress_value"])
bas_stress["stress_tertile"]  = terts
bas_stress["stress_quartile"] = quarts

# -------------------------
# Load sectors + attach basin + stress
# -------------------------
ai  = load_points_any(AI_FILE, "AI")

tri_try = pd.read_csv(TRI_FILE, low_memory=False)
tri_cols = [c.strip() for c in tri_try.columns.astype(str)]
if ("12. LATITUDE" in tri_cols) and ("13. LONGITUDE" in tri_cols):
    tri = load_points_any(TRI_FILE, "TRI", force_lat="12. LATITUDE", force_lon="13. LONGITUDE")
else:
    tri = load_points_any(TRI_FILE, "TRI")

pwr = load_points_any(POWER_FILE, "Power", force_lat="Latitude", force_lon="Longitude", excel_header=1)

fac_all = []
basins_by_sector = {}

for gdf in [ai, pwr, tri]:
    sec = gdf["sector"].iloc[0]
    joined = safe_sjoin(
        gdf.to_crs(bas_stress.crs),
        bas_stress[["PFAF_ID","stress_value","stress_tertile","stress_quartile","geometry"]],
        how="left", predicate="within"
    ).rename(columns={"PFAF_ID":"pfaf_id_lev08"})

    joined = joined.dropna(subset=["pfaf_id_lev08","stress_value","stress_tertile","stress_quartile"]).copy()
    joined["sector"] = sec
    fac_all.append(joined)
    basins_by_sector[sec] = joined["pfaf_id_lev08"].astype("Int64").dropna().unique()

fac = pd.concat(fac_all, ignore_index=True)

def run_mode(mode_name):
    mode = mode_name.upper()
    if mode not in ["GLOBAL","BASIN_MATCHED"]:
        raise ValueError("mode must be 'GLOBAL' or 'BASIN_MATCHED'")

    OUT_FIG    = OUT_DIR / f"analysis4_fig_{mode}_tertiles_quartiles_3x2.png"
    OUT_JOINED = OUT_DIR / f"analysis4_facilities_with_stress_{mode}.csv"
    OUT_RANDOM = OUT_DIR / f"analysis4_random_with_stress_{mode}.csv"
    OUT_STATS  = OUT_DIR / f"analysis4_stats_chi2_OR_{mode}.csv"

    fac.drop(columns=["index_right"], errors="ignore").to_csv(OUT_JOINED, index=False)

    rand_rows = []
    for sec in SECTORS:
        obs = fac[fac["sector"] == sec].copy()
        N = len(obs)
        if N == 0:
            continue

        poly = bas_stress.copy()
        if mode == "BASIN_MATCHED":
            use_ids = set(basins_by_sector[sec].tolist())
            poly = bas_stress[bas_stress["PFAF_ID"].isin(use_ids)].copy()

        pool_n = int(min(POOL_CAP, max(N + 1000, np.ceil(POOL_FACTOR * N))))
        seed = BASE_SEED + SEED_MAP.get(sec, 999) + MODE_SEED[mode]

        pool = generate_random_points_in_polygonset(poly, n=pool_n, seed=seed)
        poolj = safe_sjoin(
            pool,
            bas_stress[["PFAF_ID","stress_value","stress_tertile","stress_quartile","geometry"]],
            how="left", predicate="within"
        ).rename(columns={"PFAF_ID":"pfaf_id_lev08"})

        pool_valid = poolj.dropna(subset=["stress_value","stress_tertile","stress_quartile"]).copy()
        pool_valid = stable_sort_for_repro(pool_valid)

        if len(pool_valid) < N:
            raise RuntimeError(f"{sec} ({mode}): pool_valid={len(pool_valid)} < N={N}. Increase POOL_FACTOR/POOL_CAP.")

        rnd = pool_valid.iloc[:N].copy()
        rnd["sector"] = sec
        rand_rows.append(rnd)

    rand = pd.concat(rand_rows, ignore_index=True)
    rand.drop(columns=["index_right"], errors="ignore").to_csv(OUT_RANDOM, index=False)

    order3 = ["low","medium","high"]
    order4 = ["Q1","Q2","Q3","Q4"]

    fig, axes = plt.subplots(3, 2, figsize=(15.2, 9.6), dpi=160, sharey=True, constrained_layout=False)
    fig.subplots_adjust(left=0.11, right=0.99, top=0.90, bottom=0.10, wspace=0.12, hspace=0.30)

    try:
        fig.supylabel("Proportion of facilities", x=0.04, fontsize=13, fontweight="semibold")
    except Exception:
        fig.text(0.04, 0.5, "Proportion of facilities", va="center", rotation="vertical", fontsize=13, fontweight="semibold")

    for r in range(3):
        axes[r, 1].tick_params(labelleft=False)

    stats_rows = []

    for i, sec in enumerate(SECTORS):
        obs = fac[fac["sector"] == sec].copy()
        rr  = rand[rand["sector"] == sec].copy()
        N = len(obs)
        if N == 0:
            axes[i,0].axis("off"); axes[i,1].axis("off")
            continue

        obs3 = obs["stress_tertile"].value_counts().reindex(order3, fill_value=0)
        rnd3 = rr["stress_tertile"].value_counts().reindex(order3, fill_value=0)
        chi2_3, p_3, _, _ = chi2_contingency(np.vstack([obs3.values, rnd3.values]))

        a = int(obs3["high"]); b = int(obs3["low"] + obs3["medium"])
        c = int(rnd3["high"]); d = int(rnd3["low"] + rnd3["medium"])
        ORh, ORh_lo, ORh_hi = odds_ratio_2x2(a,b,c,d)

        plot_grouped_percent_only(axes[i,0], order3, obs3, rnd3, COL3, f"{sec}: Stress tertiles ({mode})")
        axes[i,0].text(
            0.02, 0.98,
            f"$\\chi^2$={chi2_3:.1f}, p={p_3:.1e}\nOR(high)={ORh:.2f} [{ORh_lo:.2f},{ORh_hi:.2f}]",
            transform=axes[i,0].transAxes, ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.95, edgecolor="black", linewidth=0.9),
            fontsize=9
        )

        obs4 = obs["stress_quartile"].value_counts().reindex(order4, fill_value=0)
        rnd4 = rr["stress_quartile"].value_counts().reindex(order4, fill_value=0)
        chi2_4, p_4, _, _ = chi2_contingency(np.vstack([obs4.values, rnd4.values]))

        a4 = int(obs4["Q4"]); b4 = int(obs4["Q1"] + obs4["Q2"] + obs4["Q3"])
        c4 = int(rnd4["Q4"]); d4 = int(rnd4["Q1"] + rnd4["Q2"] + rnd4["Q3"])
        OR4, OR4_lo, OR4_hi = odds_ratio_2x2(a4,b4,c4,d4)

        plot_grouped_percent_only(axes[i,1], order4, obs4, rnd4, COL4, f"{sec}: Stress quartiles ({mode})")
        axes[i,1].text(
            0.02, 0.98,
            f"$\\chi^2$={chi2_4:.1f}, p={p_4:.1e}\nOR(Q4)={OR4:.2f} [{OR4_lo:.2f},{OR4_hi:.2f}]",
            transform=axes[i,1].transAxes, ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.95, edgecolor="black", linewidth=0.9),
            fontsize=9
        )

        stats_rows.append({
            "Mode": mode,
            "Sector": sec,
            "N_with_stress": int(N),
            "Chi2_tertiles": float(chi2_3),
            "p_tertiles": float(p_3),
            "OR_high": float(ORh),
            "OR_high_lo": float(ORh_lo),
            "OR_high_hi": float(ORh_hi),
            "Chi2_quartiles": float(chi2_4),
            "p_quartiles": float(p_4),
            "OR_Q4": float(OR4),
            "OR_Q4_lo": float(OR4_lo),
            "OR_Q4_hi": float(OR4_hi),
        })

    from matplotlib.patches import Patch
    fig.legend(handles=[
        Patch(facecolor="lightgray", edgecolor="black", alpha=0.95, label="Facilities"),
        Patch(facecolor="lightgray", edgecolor="black", alpha=0.55, label=f"Random ({mode}, N matched)")
    ], loc="lower center", ncol=2, frameon=False)

    fig.suptitle(
        "MAIN: Basin-scale water-stress siting (GLOBAL + BASIN-MATCHED random baselines)\n"
        "Stress: Aqueduct bws_01_raw via HydroBASINS PFAF6 | Bins: GLOBAL basin quantiles | Random N matched per sector",
        y=0.98, fontsize=14, fontweight="semibold"
    )

    fig.tight_layout(rect=(0.06, 0.12, 0.995, 0.92))
    fig.savefig(OUT_FIG, dpi=350, bbox_inches="tight", pad_inches=0.20)
    plt.show()

    stats_df = pd.DataFrame(stats_rows)
    stats_df.to_csv(OUT_STATS, index=False)

    return str(OUT_FIG), str(OUT_JOINED), str(OUT_RANDOM), str(OUT_STATS), stats_df

fig_g, join_g, rand_g, stats_g, df_g = run_mode("GLOBAL")
fig_b, join_b, rand_b, stats_b, df_b = run_mode("BASIN_MATCHED")

pd.concat([df_g, df_b], ignore_index=True)